# 51 — Instrument Parent-Chain and the NCIt parent check

Tracks the `BC_Parents` hierarchy for instrument-scope BCs (domains QS, FT, RS), the
companion to `50_instrument_category_resolution.ipynb`. Where 50 covers the category side of
the section 5 identifier asymmetry, this notebook covers the hierarchy side.

**Why.** CDISC has stated the convention for `parent_bc_id` (BCD-87, 2026-11): it is the NCIt
direct parent of the concept. NCI's hierarchy does not always follow how CDISC uses a concept, so
grouping for search lives in `bc_categories` (later BC Grouping). The grouping tier added in
2026-05-26 (C222259 "Cognitive Assessment Tool", C222260 "Clinical or Research Functional
Assessment Tool") will therefore not be wired to C81250 "Functional Assessment": NCIt places both
under C20993, which is not a BC. See `docs/COSMoS_Instrument_Layer.md` §2 and §8.

This notebook does two things:

- makes the parent chains of the instrument-scope BCs explicit and diffable (Parent_Chains, Parents);
- checks the convention for every BC against NCIt FLAT: is `parent_bc_id` an NCIt direct parent of
  the BC's concept (NCIt_Parent_Check)? It states whether the published values match; it does not
  judge which side is right.

**Design.** Deterministic walk of `BC_Parents` (single-parent tree, no fabrication). Output
ships parent chains and a per-parent terminal flag, the NCIt parent check, plus a guarded diff
against a prior run. `bc_id` is read as the NCIt code, as COSMoS publishes it; a BC whose code is
not in the FLAT file is reported, not matched.

**Inputs / outputs.**
- Reads `interim/COSMoS_Graph.xlsx` (sheets `BC`, `DSS`, `BC_Parents`, `BC_Categories`).
- Reads `sdtm-test-codes/downloads/Thesaurus.txt` (NCIt FLAT, gitignored) and `Thesaurus.FLAT.zip`
  for its build date; stops if that differs from the `NCIt` pin in `Reference_Versions.json`.
- Writes `reports/Instrument_Parent_Chain.xlsx`.


In [ ]:
import json
import zipfile
import pandas as pd
from pathlib import Path
from datetime import date, datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

BASE_DIR = Path.cwd().parent          # cosmos-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/
GRAPH_FILE = BASE_DIR / 'interim' / 'COSMoS_Graph.xlsx'
REPORTS_DIR = BASE_DIR / 'reports'
OUTPUT_FILE = REPORTS_DIR / 'Instrument_Parent_Chain.xlsx'
VERSIONS = json.loads((REPO_ROOT / 'Reference_Versions.json').read_text(encoding='utf-8'))
FLAT_ZIP = REPO_ROOT / 'sdtm-test-codes' / 'downloads' / 'Thesaurus.FLAT.zip'
FLAT_FILE = REPO_ROOT / 'sdtm-test-codes' / 'downloads' / 'Thesaurus.txt'

INSTR_DOMAINS = {'QS', 'FT', 'RS'}

# Optional: path to a previous Instrument_Parent_Chain.xlsx for the version-bump diff in the
# final cell. Leave as None to skip.
PRIOR_FILE = None

for path in [GRAPH_FILE, FLAT_ZIP, FLAT_FILE]:
    if not path.exists():
        raise FileNotFoundError(f'{path} not found')
with zipfile.ZipFile(FLAT_ZIP) as zf:
    _info = zf.getinfo('Thesaurus.txt')
FLAT_DATE = date(*_info.date_time[:3]).isoformat()
if _info.file_size != FLAT_FILE.stat().st_size:
    raise RuntimeError(f'{FLAT_FILE.name} is not the file in {FLAT_ZIP.name}: unzip again')
if FLAT_DATE != VERSIONS['NCIt']:
    raise RuntimeError(f"NCIt pin {VERSIONS['NCIt']} != FLAT build date {FLAT_DATE}")

print('Graph: ', GRAPH_FILE.relative_to(REPO_ROOT))
print('NCIt FLAT build date:', FLAT_DATE, '(= pin)')
print('Output:', OUTPUT_FILE.relative_to(REPO_ROOT))


## Population and parent map

An instrument-scope BC is one that **either** has a Dataset Specialization in an instrument
domain (QS, FT, RS) **or** carries the `QRS` search category. The DSS condition catches the
question/test BCs (including RS response-criteria that are not QRS-tagged); the QRS condition
catches the `full_no_ds` family-container tier (C115789, and the new C222259/C222260), which
has no DSS and sits in a parallel `BC_Parents` chain. `BC_Parents` is a single-parent tree; a
parent not itself a child in `BC_Parents` is **terminal** (a root). The C222259/C222260 tier
is terminal by the convention: their NCIt direct parent is not a BC.


In [ ]:
bc  = pd.read_excel(GRAPH_FILE, sheet_name='BC')
dss = pd.read_excel(GRAPH_FILE, sheet_name='DSS')
bp  = pd.read_excel(GRAPH_FILE, sheet_name='BC_Parents')
bcc = pd.read_excel(GRAPH_FILE, sheet_name='BC_Categories')

short_by_bc = bc.set_index('bc_id')['bc_short_name'].to_dict()

# single-parent map (assert the tree assumption, fail fast if violated)
multi = bp.groupby('bc_id')['parent_bc_id'].nunique()
assert (multi <= 1).all(), f'BC_Parents is not single-parent: {multi[multi>1].index.tolist()}'
parent_of = dict(zip(bp['bc_id'], bp['parent_bc_id']))

dss_seed = set(dss.loc[dss['domain'].isin(INSTR_DOMAINS), 'bc_id'])
qrs_seed = set(bcc.loc[bcc['category'] == 'QRS', 'bc_id'])   # QRS search category = instrument grouping marker
seed = sorted(dss_seed | qrs_seed)
print('seed: DSS-bearing', len(dss_seed), '| QRS-category', len(qrs_seed), '| union', len(seed))

def chain(bc_id):
    path, seen = [], set()
    cur = bc_id
    while cur in parent_of and cur not in seen:
        seen.add(cur)
        cur = parent_of[cur]
        path.append(cur)
    return path  # ordered direct-parent -> root

def is_terminal(bc_id):
    return bc_id not in parent_of


## Build chains and per-parent terminal status


In [ ]:
rows = []
for b in seed:
    ch = chain(b)
    rows.append({
        'bc_id': b,
        'bc_short_name': short_by_bc.get(b, ''),
        'parent_bc_id': ch[0] if ch else '',
        'parent_short_name': short_by_bc.get(ch[0], '') if ch else '',
        'root_bc_id': ch[-1] if ch else b,
        'root_short_name': short_by_bc.get(ch[-1] if ch else b, ''),
        'chain_depth': len(ch),
        'chain_path': ' > '.join([b] + ch),
    })
parent_chains = pd.DataFrame(rows).sort_values('bc_id').reset_index(drop=True)

# distinct parents referenced by instrument BCs (direct parents and ancestors)
referenced = set()
for b in seed:
    referenced.update(chain(b))
prows = []
for p in sorted(referenced):
    in_scope_children = sum(1 for b in seed if (chain(b)[:1] == [p]))
    prows.append({
        'parent_bc_id': p,
        'parent_short_name': short_by_bc.get(p, ''),
        'is_terminal': is_terminal(p),
        'instrument_children_direct': in_scope_children,
    })
parents = pd.DataFrame(prows).sort_values(['is_terminal','parent_bc_id'], ascending=[False,True]).reset_index(drop=True)
parent_chains.head(8)


## Summary


In [ ]:
print('instrument-scope BCs:        ', len(parent_chains))
print('distinct parents referenced:  ', len(parents))
print('terminal parents (roots):     ', int(parents['is_terminal'].sum()))
print('max chain depth:              ', int(parent_chains['chain_depth'].max()))
print()
print('Terminal parents with instrument children:')
print(parents[parents['is_terminal'] & (parents['instrument_children_direct']>0)]
      [['parent_bc_id','parent_short_name','instrument_children_direct']].to_string(index=False))


## NCIt parent check (all BCs)

One row per BC. `ncit_parents` are the NCIt direct parents of the BC's concept in the FLAT file,
as published (`|`-separated). `status` says whether `parent_bc_id` is one of them; for a BC
without `parent_bc_id`, whether one of its NCIt direct parents is itself a BC.
`parent_is_ncit_ancestor` says, where `parent_bc_id` is not a direct parent, whether it is a
higher NCIt ancestor. Facts only: the check reports where the two disagree, it does not repair.

In [ ]:
flat = pd.read_csv(FLAT_FILE, sep='\t', header=None, usecols=[0, 2], names=['code', 'parents'],
                   dtype=str, keep_default_na=False, quoting=3)
ncit_parents = {c: [p for p in ps.split('|') if p] for c, ps in zip(flat['code'], flat['parents'])}

def ncit_ancestors(code):
    seen, stack = set(), list(ncit_parents.get(code, []))
    while stack:
        c = stack.pop()
        if c not in seen:
            seen.add(c)
            stack.extend(ncit_parents.get(c, []))
    return seen

bc_ids = set(bc['bc_id'])
type_by_bc = bc.set_index('bc_id')['bc_type'].to_dict()
seed_set = set(seed)
crows = []
for b in sorted(bc_ids):
    nps = ncit_parents.get(b)
    p = parent_of.get(b, '')
    if nps is None:
        status = 'concept not in NCIt FLAT'
    elif p:
        status = 'parent_bc_id is an NCIt direct parent' if p in nps else 'parent_bc_id is not an NCIt direct parent'
    elif any(n in bc_ids for n in nps):
        status = 'no parent_bc_id; an NCIt direct parent is a BC'
    else:
        status = 'no parent_bc_id; no NCIt direct parent is a BC'
    crows.append({
        'bc_id': b,
        'bc_short_name': short_by_bc.get(b, ''),
        'bc_type': type_by_bc.get(b, ''),
        'instrument_scope': b in seed_set,
        'parent_bc_id': p,
        'ncit_parents': '|'.join(nps or []),
        'ncit_parents_that_are_bcs': '|'.join(n for n in (nps or []) if n in bc_ids),
        'parent_is_ncit_ancestor': (p in ncit_ancestors(b)) if (p and nps is not None and p not in nps) else '',
        'status': status,
    })
ncit_check = pd.DataFrame(crows)
print(pd.crosstab(ncit_check['status'], ncit_check['instrument_scope'], margins=True))

## Write report

`reports/Instrument_Parent_Chain.xlsx` — README, Parent_Chains, Parents, NCIt_Parent_Check sheets. Yellow
(`FFD700`) headers for COSMoS-layer columns, grey (`808080`) for key columns.


In [ ]:
REPORTS_DIR.mkdir(exist_ok=True)

YELLOW = PatternFill('solid', fgColor='FFD700')
GREY   = PatternFill('solid', fgColor='808080')
HDR_BLACK = Font(bold=True, color='000000')
HDR_WHITE = Font(bold=True, color='FFFFFF')
BODY = Font(size=11)

wb = Workbook()
ws = wb.active
ws.title = 'README'
readme = [
    ('Instrument Parent-Chain', Font(bold=True, size=14)),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', BODY),
    ('', BODY),
    ('BC_Parents hierarchy for instrument-scope BCs (DSS in QS/FT/RS).', BODY),
    ('Source: cosmos-graph/interim/COSMoS_Graph.xlsx (BC, DSS, BC_Parents, BC_Categories); NCIt FLAT.', BODY),
    (f'Source versions: COSMoS {VERSIONS["COSMoS"]}, NCIt FLAT {FLAT_DATE}.', BODY),
    ('', BODY),
    ('Parent_Chains: one row per instrument BC.', Font(bold=True, size=11)),
    ('  bc_id, parent_bc_id, root_bc_id - keys.', BODY),
    ('  chain_depth - hops from BC to root. chain_path - full BC > ... > root.', BODY),
    ('Parents: distinct parents referenced by instrument BCs.', Font(bold=True, size=11)),
    ('  is_terminal - parent has no parent of its own (a root).', BODY),
    ('  instrument_children_direct - instrument BCs whose direct parent is this node.', BODY),
    ('', BODY),
    ('NCIt_Parent_Check: one row per BC (all BCs, not only instrument scope).', Font(bold=True, size=11)),
    ('  CDISC convention (BCD-87, 2026-11): parent_bc_id is the NCIt direct parent of the concept.', BODY),
    ('  ncit_parents - NCIt direct parents in NCIt FLAT, as published, | separated.', BODY),
    ('  status - whether parent_bc_id is one of them; without parent_bc_id, whether an NCIt direct parent is a BC.', BODY),
    ('  parent_is_ncit_ancestor - where parent_bc_id is not a direct parent: is it a higher NCIt ancestor.', BODY),
    ('  bc_id is read as the NCIt code; a BC whose code is not in the FLAT file is reported, not matched.', BODY),
    ('', BODY),
    ('Companion to 50_instrument_category_resolution (category side). See Instrument_Layer.md s2 and s8.', BODY),
]
for i,(t,f) in enumerate(readme, start=1):
    c = ws.cell(row=i, column=1, value=t); c.font = f
ws.column_dimensions['A'].width = 88

def write_sheet(df, title, key_cols):
    ws = wb.create_sheet(title)
    for j,col in enumerate(df.columns, start=1):
        c = ws.cell(row=1, column=j, value=col)
        if col in key_cols:
            c.fill = GREY; c.font = HDR_WHITE
        else:
            c.fill = YELLOW; c.font = HDR_BLACK
        c.alignment = Alignment(horizontal='left')
    for i,(_,row) in enumerate(df.iterrows(), start=2):
        for j,col in enumerate(df.columns, start=1):
            ws.cell(row=i, column=j, value=row[col])
    for j,col in enumerate(df.columns, start=1):
        vals = [len(str(col))] + [len(str(v)) for v in df[col]]
        ws.column_dimensions[get_column_letter(j)].width = min(max(vals)+2, 70)
    ws.freeze_panes = 'A2'

write_sheet(parent_chains, 'Parent_Chains', key_cols={'bc_id','parent_bc_id','root_bc_id'})
write_sheet(parents, 'Parents', key_cols={'parent_bc_id'})
write_sheet(ncit_check, 'NCIt_Parent_Check', key_cols={'bc_id', 'parent_bc_id'})

wb.save(OUTPUT_FILE)
print('Wrote', OUTPUT_FILE.relative_to(REPO_ROOT))


## Version-bump diff (optional)

Set `PRIOR_FILE` (cell 2) to a previous `Instrument_Parent_Chain.xlsx` to compare. Reports
reparented BCs, new parents, and terminal-status flips (the "did CDISC wire up the new tier"
check). No-ops if unset or missing.


In [ ]:
if PRIOR_FILE and Path(PRIOR_FILE).exists():
    pc_old = pd.read_excel(PRIOR_FILE, sheet_name='Parent_Chains').fillna('')
    pr_old = pd.read_excel(PRIOR_FILE, sheet_name='Parents').fillna('')
    op = dict(zip(pc_old['bc_id'], pc_old['parent_bc_id']))
    np_ = dict(zip(parent_chains['bc_id'], parent_chains['parent_bc_id']))
    reparented = [(b, op.get(b,''), np_[b]) for b in sorted(set(op)&set(np_)) if op.get(b,'')!=np_[b]]
    new_parents = sorted(set(parents['parent_bc_id']) - set(pr_old['parent_bc_id']))
    term_old = dict(zip(pr_old['parent_bc_id'], pr_old['is_terminal']))
    term_flip = [(p, bool(term_old[p]), bool(t)) for p,t in
                 zip(parents['parent_bc_id'], parents['is_terminal'])
                 if p in term_old and bool(term_old[p]) != bool(t)]
    print(f'Prior: {PRIOR_FILE}')
    print(f'Reparented instrument BCs ({len(reparented)}):')
    for b,a,c in reparented:
        print(f'  {b}: {a or "(none)"} -> {c}')
    print(f'New parents in instrument scope: {new_parents}')
    print(f'Terminal-status flips (parent: prior_is_terminal -> now):')
    for p,a,c in term_flip:
        print(f'  {p}: {a} -> {c}')
    if not (reparented or new_parents or term_flip):
        print('  (no hierarchy changes)')
else:
    print('PRIOR_FILE not set or missing - skipping version diff.')
